In [1]:
import time, os, sqlite3 

DB_PATH = "experiment.db"
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

setup = sqlite3.connect(DB_PATH, autocommit=False)
setup.execute("""
    CREATE TABLE clients (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT NOT NULL,
        CONSTRAINT uq_client_name UNIQUE (name)
    )
""")

setup.execute("INSERT INTO clients (name) VALUES (?)", ("Farm A",))
setup.commit()
setup.close()

w1 = sqlite3.connect(DB_PATH, autocommit=False, timeout=0.5)
w2 = sqlite3.connect(DB_PATH, autocommit=False, timeout=0.5)

def attempt(conn, label, sql, params):
    start = time.perf_counter()
    try:
        conn.execute(sql, params)
        print(f"{label}: statement OK after {time.perf_counter() - start: .2f}s")
        return True 
    except sqlite3.Error as err:
        print(f"{label}: {type(err).__name__} after {time.perf_counter() - start: .2f}s")
        print(f"    message: {err} | errorname: {getattr(err, 'sqlite_errorname', None)} | errorcode: {getattr(err, 'sqlite_errorcode', None)}")
        return False

In [4]:
def rows(conn):
    return conn.execute("SELECT id, name FROM clients ORDER BY id").fetchall()

In [2]:
attempt(w1, "w1 insert Farm B", "INSERT INTO clients (name) VALUES (?)", ("Farm B",))

w1 insert Farm B: statement OK after  0.00s


True

In [3]:
attempt(w2, "w2 insert Farm C", "INSERT INTO clients (name) VALUES (?)", ("Farm C",))
print("w2 in_transaction:", w2.in_transaction)
w2.rollback()

w2 insert Farm C: OperationalError after  0.58s
    message: database is locked | errorname: SQLITE_BUSY | errorcode: 5
w2 in_transaction: True


In [5]:
w1.commit()
if attempt(w2, "w2 insert Farm C", "INSERT INTO clients (name) VALUES (?)", ("Farm C",)):
    w2.commit()
print("rows: ", rows(w2))
w2.rollback() # end w2's read transaction

w2 insert Farm C: statement OK after  0.00s
rows:  [(1, 'Farm A'), (2, 'Farm B'), (3, 'Farm C')]


In [7]:
attempt(w1, "w1 insert Farm D", "INSERT INTO clients (name) VALUES (?)", ("Farm D",))
attempt(w2, "w2 insert Farm D (while w1 open)", "INSERT INTO clients (name) VALUES (?)", ("Farm D",))
w2.rollback()

w1.commit()
attempt(w2, "w2 retry Farm D (after w1 commit)", "INSERT INTO clients (name) VALUES (?)", ("Farm D",))
w2.rollback()

w1 insert Farm D: statement OK after  0.00s
w2 insert Farm D (while w1 open): OperationalError after  0.57s
    message: database is locked | errorname: SQLITE_BUSY | errorcode: 5
w2 retry Farm D (after w1 commit): IntegrityError after  0.00s
    message: UNIQUE constraint failed: clients.name | errorname: SQLITE_CONSTRAINT_UNIQUE | errorcode: 2067


In [9]:
attempt(w1, "w1 insert Farm F", "INSERT INTO clients (name) VALUES (?)", ("Farm F",))
attempt(w2, "w2 insert Farm F (while w1 open)", "INSERT INTO clients (name) VALUES (?)", ("Farm F",))
w2.rollback()

w1.rollback()
attempt(w2, "w2 retry Farm F (after w1 commit)", "INSERT INTO clients (name) VALUES (?)", ("Farm F",))
w2.rollback()

w1 insert Farm F: statement OK after  0.00s
w2 insert Farm F (while w1 open): OperationalError after  0.58s
    message: database is locked | errorname: SQLITE_BUSY | errorcode: 5
w2 retry Farm F (after w1 commit): statement OK after  0.00s


In [10]:
rows(w1); rows(w2)  # both now hold read transactions

attempt(w1, "w1 insert Farm G", "INSERT INTO clients (name) VALUES (?)", ("Farm G",))
attempt(w2, "w2 insert Farm H", "INSERT INTO clients (name) VALUES (?)", ("Farm H",))

try:
    w1.commit(); print("w1 commit OK")
except sqlite3.Error as err:
    print("w1 commit: ", type(err).__name__, "-", err)

w1.rollback(); w2.rollback()

w1 insert Farm G: statement OK after  0.00s
w2 insert Farm H: OperationalError after  0.00s
    message: database is locked | errorname: SQLITE_BUSY | errorcode: 5
w1 commit:  OperationalError - database is locked


In [11]:
w1.close()
w2.close()